# Deployment and reproducibility: the model is more than its weights

A saved parameter tensor is not a complete model artifact. Inference also depends on preprocessing statistics, feature order, label names, model architecture, and dependency versions. This lesson builds a small classifier, saves that contract, reloads it in a fresh object, validates inputs, and measures the call that a service would expose.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import torch
from torch import nn

SEED=3; torch.manual_seed(SEED)
class TwoFeatureModel(nn.Module):
    def __init__(self):
        super().__init__(); self.network=nn.Sequential(nn.Linear(2,8),nn.ReLU(),nn.Linear(8,2))
    def forward(self,x): return self.network(x)

model=TwoFeatureModel(); model.eval()
metadata={'feature_names':['x1','x2'],'mean':[1.5,-.5],'scale':[2.0,4.0],'classes':['class_0','class_1'],'model_version':'demo-1'}
artifact={'state_dict':model.state_dict(),'metadata':metadata,'torch_version':str(torch.__version__)}
path=Path('deployment_demo.pt'); torch.save(artifact,path)
print(path.resolve())

## The inference contract

Validation happens at the boundary. A service should reject the wrong number of features, non-finite values, and a preprocessing scale of zero before the tensor reaches the network. This makes bad requests observable instead of silently producing a plausible-looking class.

In [ ]:
def load_artifact(path):
    saved=torch.load(path,map_location='cpu',weights_only=True); loaded=TwoFeatureModel(); loaded.load_state_dict(saved['state_dict']); loaded.eval(); return loaded,saved['metadata']

def predict(loaded,meta,values):
    values=np.asarray(values,dtype=np.float32)
    if values.shape!=(2,): raise ValueError(f'expected shape (2,), got {values.shape}')
    if not np.isfinite(values).all(): raise ValueError('features must be finite')
    scale=np.asarray(meta['scale'],dtype=np.float32)
    if (scale==0).any(): raise ValueError('scale cannot be zero')
    x=(values-np.asarray(meta['mean'],dtype=np.float32))/scale
    with torch.no_grad(): p=loaded(torch.from_numpy(x).unsqueeze(0)).softmax(1)[0]
    index=int(p.argmax()); return {'label':meta['classes'][index],'probabilities':p.numpy()}

loaded,meta=load_artifact(path)
print(predict(loaded,meta,[2.0,1.0]))
for bad in ([1.0], [1.0,float('nan')]):
    try: predict(loaded,meta,bad)
    except ValueError as error: print('rejected:',error)

## Round-trip and latency checks

A round-trip test compares predictions before and after saving. A latency measurement should specify batch size and warm-up policy; a single cold call includes loading and compilation effects that a long-running service does not.

In [ ]:
examples=np.array([[2.,1.],[-1.,.5],[3.,-2.]],dtype=np.float32)
before=np.stack([predict(model,metadata,row)['probabilities'] for row in examples])
after=np.stack([predict(loaded,meta,row)['probabilities'] for row in examples])
print('round-trip identical:',np.allclose(before,after,atol=1e-7))
for _ in range(5): predict(loaded,meta,examples[0])
start=time.perf_counter();
for row in examples: predict(loaded,meta,row)
elapsed=(time.perf_counter()-start)/len(examples)*1000
print(f'mean single-example latency after warm-up: {elapsed:.3f} ms')
path.unlink()

### What still belongs outside this toy artifact

Production deployment needs dependency locking, request logging, monitoring for drift, access control, load testing, and a rollback path. The important lesson is the boundary: preprocessing and metadata are part of the model interface, not comments that can be reconstructed later.